# Importing modules and settings

### Importing libraries

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from matplotlib.pyplot import rc_context

General settings of Scanpy

In [ ]:
sc.settings.verbosity = 3 
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white')

In [ ]:
results_file = './Smed_L78-L47_20250523_Ingest.h5ad'

# Ingest with the 1st dataset

## Reading the reference dataset

In [ ]:
# Read the reference dataset: 
# Multiplex single-cell analysis of serotonergic neuron function in planarians reveals widespread effects in diverse cell types
# Elena Emili, Dianalí Rodríguez-Fernández, Alberto Pérez-Posada, Helena García-Castro, Jordi Solana
# bioRxiv 2024.02.28.581916; doi: https://doi.org/10.1101/2024.02.28.581916
# GEO GSE256032
adata_ref = sc.read_h5ad('../reference/Serotonergic_data_annotated.h5ad')

In [ ]:
adata_ref

In [ ]:
# View columns
adata_ref.obs.columns

In [ ]:
# Assign the `adata_ref.obs` column that contains the labels
labels = 'leiden_3_names'

In [ ]:
ref_name = 'sero'

In [ ]:
# UMAP of adata_ref
sc.pl.umap(adata_ref, color= labels)

In [ ]:
# Return the number of labels in the references dataset
len(adata_ref.obs[labels].cat.categories)

In [ ]:
# Return the labels in the reference dataset
adata_ref.obs[labels].cat.categories

## Reading the query dataset

In [ ]:
# Read the query dataset
adata = sc.read_h5ad('./Smed_L78-L47_20250523_Results.h5ad')

In [ ]:
# Add the names of each Leiden resolution to a list
leiden_names = adata.obs.columns[adata.obs.columns.str.contains('leiden')].to_list()

In [ ]:
leiden_names

In [ ]:
# Make a copy of the adata object and store it in a new variable
adata_ingest = adata.copy()
adata_ingest

In [ ]:
# Create a list of the genes that are common to both the reference and the query datasets
var_names = adata_ref.var_names.intersection(adata_ingest.var_names)
var_names

In [ ]:
# Slice both datasets by the common genes
adata_ref = adata_ref[:, var_names]
adata_ingest = adata_ingest[:, var_names]

In [ ]:
# View the number of cells and genes in the newly selected dataset.
adata_ingest.shape

In [ ]:
# View the number of cells and genes in the reference dataset
adata_ref.shape

In [ ]:
# View the number of cells and genes in the adata object
adata.shape

## Ingest label transfer

In [ ]:
# Run the Ingest algorithm
sc.tl.ingest(adata_ingest, adata_ref, obs= labels, embedding_method=('umap', 'pca'), labeling_method='knn', neighbors_key=None, inplace=True)

In [ ]:
adata_ingest.obs

In [ ]:
# Copy the ingested labels from the adata_ingest copy to the original query dataset, adata, by creating a new coloumn
adata.obs['ingest_' + ref_name + '_' + labels] = adata_ingest.obs[labels]

In [ ]:
adata.obs

In [ ]:
# View the number of clusters transferred successfully
len(adata.obs['ingest_' + ref_name + '_' + labels].cat.categories)

In [ ]:
adata.obs['ingest_' + ref_name + '_' + labels].cat.categories

**Add reference cluster colours to adata**

In [ ]:
list(adata_ref.uns)

In [ ]:
# Make a list of the cluster colours from the adata reference
cluster_colours = adata_ref.uns[labels + '_colors']
cluster_colours

In [ ]:
# Transfer the colours from the Ingest
adata.uns['ingest_' + ref_name + '_' + labels + '_colors'] = cluster_colours

In [ ]:
list (adata.uns)

**Plot transfered clusters**

In [ ]:
# UMAP of transferred clusters
with rc_context({'figure.figsize': (12, 12)}):
     sc.pl.umap(adata, color=['ingest_' + ref_name + '_' + labels], legend_fontoutline = 5, title= 'Clustering ', size = 30, 
         frameon=False, add_outline = True)

In [ ]:
with rc_context({'figure.figsize': (15, 15)}):
    sc.pl.umap(adata, color=['ingest_' + ref_name + '_' + labels], legend_loc='on data', title='transferred_cell_identities', size = 30, frameon=False)

## Transfer labels to leiden clusters in different resolutions

In [ ]:
# Create a new coloumn with the combined Ingest and Leiden clusters 
for l in leiden_names:
    for i in adata.obs[l].cat.categories:
        filt = adata.obs[l] == i
        adata.obs.loc[filt, l + '_ingested_' + ref_name] = i + ' : ' + adata.obs[filt]['ingest_' + ref_name + '_' + labels].value_counts().sort_index().idxmax()
    

In [ ]:
adata.obs.columns

In [ ]:
# Plot a UMAP of the clusters
for l in leiden_names:
    with rc_context({'figure.figsize': (15, 15)}):
        sc.pl.umap(adata, color= l + '_ingested_' + ref_name , legend_loc='on data', title = l, size = 30, frameon=False)

# Ingest with the 2nd reference dataset

## Reading the reference dataset

In [ ]:
# This second run of ingest will use the sizes allometry dataset:
# Emili E, Pérez-Posada A, Vanni V, Salamanca-Díaz D, Ródriguez-Fernández D, Christodoulou MD, Solana J. 
# Allometry of cell types in planarians by single-cell transcriptomics. 
# Sci Adv. 2025 May 9;11(19):eadm7042. doi: 10.1126/sciadv.adm7042
# GEO GSE246681
adata_ref = sc.read_h5ad('../reference/smed_size_analysis_202306.h5ad')

In [ ]:
adata_ref

In [ ]:
adata_ref.obs.columns

In [ ]:
labels = 'leiden_3_names'

In [ ]:
ref_name = 'sizes'

In [ ]:
sc.pl.umap(adata_ref, color= labels)

In [ ]:
len(adata_ref.obs[labels].cat.categories)

In [ ]:
adata_ref.obs[labels].cat.categories

## Reading the query dataset

In [ ]:
adata.obs

In [ ]:
leiden_names = adata.obs.columns[adata.obs.columns.str.contains('leiden') & ~adata.obs.columns.str.contains('ingest')].to_list()

In [ ]:
leiden_names

In [ ]:
adata_ingest = adata.copy()
adata_ingest

In [ ]:
var_names = adata_ref.var_names.intersection(adata_ingest.var_names)
var_names

In [ ]:
adata_ref = adata_ref[:, var_names]
adata_ingest = adata_ingest[:, var_names]

In [ ]:
adata_ingest.shape

In [ ]:
adata_ref.shape

In [ ]:
adata.shape

## Ingest label transfer

In [ ]:
sc.tl.ingest(adata_ingest, adata_ref, obs= labels, embedding_method=('umap', 'pca'), labeling_method='knn', neighbors_key=None, inplace=True)

In [ ]:
adata_ingest.obs

In [ ]:
adata.obs['ingest_' + ref_name + '_' + labels] = adata_ingest.obs[labels]

In [ ]:
adata.obs

In [ ]:
len(adata.obs['ingest_' + ref_name + '_' + labels].cat.categories)

In [ ]:
adata.obs['ingest_' + ref_name + '_' + labels].cat.categories

**Add reference cluster colours to adata**

In [ ]:
list(adata_ref.uns)

In [ ]:
cluster_colours = adata_ref.uns[labels + '_colors']
cluster_colours

In [ ]:
adata.uns['ingest_' + ref_name + '_' + labels + '_colors'] = cluster_colours

In [ ]:
list (adata.uns)

**Plot transfered clusters**

In [ ]:
with rc_context({'figure.figsize': (12, 12)}):
     sc.pl.umap(adata, color=['ingest_' + ref_name + '_' + labels], legend_fontoutline = 5, title= 'Clustering ', size = 30, 
         frameon=False, add_outline = True)

In [ ]:
with rc_context({'figure.figsize': (15, 15)}):
    sc.pl.umap(adata, color=['ingest_' + ref_name + '_' + labels], legend_loc='on data', title='transferred_cell_identities', size = 30, frameon=False)

## Transfer labels to leiden clusters in different resolutions

In [ ]:
for l in leiden_names:
    for i in adata.obs[l].cat.categories:
        filt = adata.obs[l] == i
        adata.obs.loc[filt, l + '_ingested_' + ref_name] = i + ' : ' + adata.obs[filt]['ingest_' + ref_name + '_' + labels].value_counts().sort_index().idxmax()
    

In [ ]:
adata.obs.columns

In [ ]:
for l in leiden_names:
    with rc_context({'figure.figsize': (15, 15)}):
        sc.pl.umap(adata, color= l + '_ingested_' + ref_name, legend_loc='on data', title = l, size = 30, frameon=False)

In [ ]:
adata.obs

# Saving changes

In [ ]:
adata.write(results_file)